In [13]:
import sys
import os
import numpy as np
import pandas as pd
import pickle
import time

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Add src/ to path so we can import shared utilities
sys.path.append(os.path.join(os.getcwd(), '..', '..'))
from src.preprocessing import apply_smote
from src.evaluation import compute_metrics, plot_confusion_matrix, save_results
# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [14]:
# ## 2. DNN Model Architecture

# %%
class IntrusionDetectionDNN(nn.Module):
    """
    Fully connected feedforward network for 13-class classification.

    Architecture:
        Input → 128 → BatchNorm → ReLU → Dropout
              → 64  → BatchNorm → ReLU → Dropout
              → 32  → BatchNorm → ReLU → Dropout
              → 13 (output)

    Input size changes based on feature set (10, 20, 30, 83, or PCA components).
    Output is always 13 (one per class).
    """

    def __init__(self, input_size, num_classes=13, dropout_rate=0.3):
        super(IntrusionDetectionDNN, self).__init__()

        self.network = nn.Sequential(
            # Layer 1: Input → 128
            nn.Linear(input_size, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            # Layer 2: 128 → 64
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            # Layer 3: 64 → 32
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            # Output layer: 32 → 13
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.network(x)


# Quick test — model should initialize without errors
test_model = IntrusionDetectionDNN(input_size=83)
print(test_model)
print(f"\nTotal parameters: {sum(p.numel() for p in test_model.parameters()):,}")
del test_model


IntrusionDetectionDNN(
  (network): Sequential(
    (0): Linear(in_features=83, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.3, inplace=False)
    (12): Linear(in_features=32, out_features=13, bias=True)
  )
)

Total parameters: 21,965


In [15]:
# ## 3. Training Utilities

# %%
def create_dataloaders(X_train, y_train, X_test, y_test, batch_size=256):
    """Convert numpy arrays to PyTorch DataLoaders."""
    # Convert to tensors
    X_train_t = torch.FloatTensor(np.array(X_train)).to(device)
    y_train_t = torch.LongTensor(np.array(y_train)).to(device)
    X_test_t = torch.FloatTensor(np.array(X_test)).to(device)
    y_test_t = torch.LongTensor(np.array(y_test)).to(device)

    # Create datasets
    train_dataset = TensorDataset(X_train_t, y_train_t)
    test_dataset = TensorDataset(X_test_t, y_test_t)

    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader


def compute_class_weights(y_train):
    """
    Compute inverse class frequency weights for cost-sensitive learning.
    Used in CrossEntropyLoss(weight=...) to penalize minority class errors more.

    Example: NMAP_FIN_SCAN (~22 samples) gets a much higher weight than
    DOS_SYN_Hping (~75,000 samples), forcing the model to care about rare attacks.
    """
    classes = np.unique(y_train)
    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y_train
    )
    weights_tensor = torch.FloatTensor(weights).to(device)

    print("Class weights (higher = rarer class):")
    for cls, w in zip(classes, weights):
        print(f"  Class {cls}: {w:.4f}")


    return weights_tensor
    



In [16]:

# ## 4. Training Loop

# %%
def train_dnn(
    model,
    train_loader,
    test_loader,
    criterion,
    optimizer,
    num_epochs=100,
    patience=10,
    model_name="DNN"
):
    """
    Train the DNN with early stopping.

    Args:
        model: IntrusionDetectionDNN instance
        train_loader: training DataLoader
        test_loader: test DataLoader (used for validation/early stopping)
        criterion: loss function (CrossEntropyLoss, with or without weights)
        optimizer: Adam optimizer
        num_epochs: maximum epochs to train
        patience: stop if validation loss doesn't improve for this many epochs
        model_name: string label for logging

    Returns:
        dict with training history (losses per epoch) and best model state
    """
    print(f"\n{'='*60}")
    print(f" Training: {model_name}")
    print(f" Epochs: {num_epochs} | Patience: {patience}")
    print(f"{'='*60}")

    train_losses = []
    val_losses = []
    best_val_loss = float("inf")
    best_model_state = None
    patience_counter = 0
    start_time = time.time()

    for epoch in range(num_epochs):
        # --- Training phase ---
        model.train()
        epoch_train_loss = 0.0
        num_batches = 0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            epoch_train_loss += loss.item()
            num_batches += 1

        avg_train_loss = epoch_train_loss / num_batches
        train_losses.append(avg_train_loss)

        # --- Validation phase ---
        model.eval()
        epoch_val_loss = 0.0
        num_val_batches = 0

        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                epoch_val_loss += loss.item()
                num_val_batches += 1

        avg_val_loss = epoch_val_loss / num_val_batches
        val_losses.append(avg_val_loss)

        # --- Early stopping check ---
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1

        # Print every 10 epochs
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:3d}/{num_epochs} | "
                  f"Train Loss: {avg_train_loss:.4f} | "
                  f"Val Loss: {avg_val_loss:.4f} | "
                  f"Patience: {patience_counter}/{patience}")

        if patience_counter >= patience:
            print(f"\n  Early stopping at epoch {epoch+1}")
            break

    elapsed = time.time() - start_time
    print(f"  Training time: {elapsed:.1f}s")

    # Load best model
    model.load_state_dict(best_model_state)

    return {
        "train_losses": train_losses,
        "val_losses": val_losses,
        "best_epoch": len(train_losses) - patience_counter,
        "training_time": elapsed,
    }
    

In [17]:

# ## 5. Prediction & Evaluation

# %%
def predict_dnn(model, test_loader):
    """Get predictions from trained model."""
    model.eval()
    all_preds = []

    with torch.no_grad():
        for X_batch, _ in test_loader:
            outputs = model(X_batch)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())

    return np.array(all_preds)


def plot_loss_curves(history, model_name="DNN", save_path=None):
    """Plot training and validation loss curves."""
    plt.figure(figsize=(10, 5))
    plt.plot(history["train_losses"], label="Train Loss", linewidth=2)
    plt.plot(history["val_losses"], label="Val Loss", linewidth=2)
    plt.axvline(
        x=history["best_epoch"],
        color="red",
        linestyle="--",
        alpha=0.7,
        label=f"Best Epoch ({history['best_epoch']})"
    )
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Training & Validation Loss — {model_name}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"  Saved: {save_path}")
    plt.show()



In [18]:

# ## 6. Run All Experiments
#
# This is the main experiment runner.
# It runs DNN across 5 feature sets × 3 imbalance strategies = 15 runs.

# %%
def run_single_experiment(
    X_train, y_train, X_test, y_test,
    feature_set_name, imbalance_strategy,
    target_names=None,
    num_epochs=100, patience=10, lr=0.001, dropout=0.3, batch_size=256
):
    """
    Run one DNN experiment with a specific feature set and imbalance strategy.

    Args:
        X_train, y_train: training data
        X_test, y_test: test data (never modified)
        feature_set_name: e.g., "full-83", "top-10", "pca"
        imbalance_strategy: "baseline", "smote", or "cost-sensitive"
        target_names: list of class names for reporting
    """
    model_name = f"DNN | {feature_set_name} | {imbalance_strategy}"
    print(f"\n{'#'*70}")
    print(f"# Experiment: {model_name}")
    print(f"{'#'*70}")

    # --- Step 1: Apply imbalance strategy ---
    if imbalance_strategy == "smote":
        print("Applying SMOTE to training data...")
        X_train_exp, y_train_exp = apply_smote(X_train, y_train)
    else:
        X_train_exp, y_train_exp = X_train, y_train

    # --- Step 2: Create dataloaders ---
    train_loader, test_loader = create_dataloaders(
        X_train_exp, y_train_exp, X_test, y_test, batch_size=batch_size
    )

    # --- Step 3: Initialize model ---
    input_size = X_train_exp.shape[1]
    model = IntrusionDetectionDNN(
        input_size=input_size,
        num_classes=13,
        dropout_rate=dropout
    ).to(device)

    # --- Step 4: Set up loss function ---
    if imbalance_strategy == "cost-sensitive":
        print("Computing class weights for cost-sensitive learning...")
        class_weights = compute_class_weights(np.array(y_train_exp))
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(model.parameters(), lr=lr)

    # --- Step 5: Train ---
    history = train_dnn(
        model, train_loader, test_loader,
        criterion, optimizer,
        num_epochs=num_epochs,
        patience=patience,
        model_name=model_name
    )

    # --- Step 6: Predict & evaluate ---
    y_pred = predict_dnn(model, test_loader)
    results = compute_metrics(y_test, y_pred)

    # Add extra info
    results["feature_set"] = feature_set_name
    results["imbalance_strategy"] = imbalance_strategy
    results["training_time"] = history["training_time"]
    results["history"] = history

    # --- Step 7: Save loss curve ---
    save_path = f"../../figures/dnn_loss_{feature_set_name}_{imbalance_strategy}.png"
    plot_loss_curves(history, model_name=model_name, save_path=save_path)

    # --- Step 8: Save model ---
    model_path = f"../../models/dnn_{feature_set_name}_{imbalance_strategy}.pt"
    torch.save(model.state_dict(), model_path)
    print(f"  Model saved: {model_path}")

    return results


In [20]:

# ## 7. Load Preprocessed Data & Run All 15 Experiments
#
# ⚠️ **WAIT**: Run this section only after Piyush has:
# 1. Saved preprocessed data to Drive (Phase 1)
# 2. Saved feature subsets to Drive (Phase 2)
#
# Update the file paths below to match your Drive/local setup.

# %%
# ============================================================
# UPDATE THESE PATHS to wherever Piyush saved the data
# ============================================================
DATA_DIR = "../../data/preprocessed"  # or Google Drive path

# Load preprocessed data
# Uncomment and update when data is ready:

X_train_full = pd.read_pickle(f"{DATA_DIR}/X_train_scaled.pkl")
X_test_full = pd.read_pickle(f"{DATA_DIR}/X_test_scaled.pkl")
y_train = pd.read_pickle(f"{DATA_DIR}/y_train.pkl")
y_test = pd.read_pickle(f"{DATA_DIR}/y_test.pkl")
target_encoder = pd.read_pickle(f"{DATA_DIR}/target_encoder.pkl")
target_names = list(target_encoder.classes_)

# Feature subsets — uncomment when Piyush pushes them:
X_train_top10 = pd.read_pickle(f"{DATA_DIR}/X_train_top10.pkl")
X_test_top10 = pd.read_pickle(f"{DATA_DIR}/X_test_top10.pkl")
X_train_top20 = pd.read_pickle(f"{DATA_DIR}/X_train_top20.pkl")
X_test_top20 = pd.read_pickle(f"{DATA_DIR}/X_test_top20.pkl")
X_train_top30 = pd.read_pickle(f"{DATA_DIR}/X_train_top30.pkl")
X_test_top30 = pd.read_pickle(f"{DATA_DIR}/X_test_top30.pkl")
X_train_pca = pd.read_pickle(f"{DATA_DIR}/X_train_pca.pkl")
X_test_pca = pd.read_pickle(f"{DATA_DIR}/X_test_pca.pkl")

print(f"Full feature set: {X_train_full.shape[1]} features")
print(f"Target classes: {target_names}")

# %%
# ============================================================
# RUN ALL 15 EXPERIMENTS
# ============================================================
# Uncomment this block when data is loaded above

feature_sets = {
    "full-83": (X_train_full, X_test_full),
    "top-30": (X_train_top30, X_test_top30),
    "top-20": (X_train_top20, X_test_top20),
    "top-10": (X_train_top10, X_test_top10),
    "pca": (X_train_pca, X_test_pca),
}

imbalance_strategies = ["baseline", "smote", "cost-sensitive"]

all_results = []

for fs_name, (X_tr, X_te) in feature_sets.items():
    for strategy in imbalance_strategies:
        result = run_single_experiment(
            X_train=X_tr,
            y_train=y_train,
            X_test=X_te,
            y_test=y_test,
            feature_set_name=fs_name,
            imbalance_strategy=strategy,
            target_names=target_names,
            num_epochs=100,
            patience=10,
            lr=0.001,
            dropout=0.3,
            batch_size=256,
        )
        all_results.append(result)

# # Save all results
save_results(all_results, "../../results/mahip_results.csv")
print("\nAll 15 DNN experiments complete!")

FileNotFoundError: [Errno 2] No such file or directory: '../../data/preprocessed/X_train_scaled.pkl'

In [ ]:
# ## 8. Quick Results Summary
#
# Uncomment after running experiments above.

# %%
results_df = pd.read_csv("../../results/mahip_results.csv")
print(results_df.sort_values("macro_f1", ascending=False).to_string(index=False))


In [ ]:

# ## 9. SMOTE Before/After Comparison (for report Figure)
#
# Run this to generate the before/after class distribution figure.

# %%
from src.data_loader import print_class_distribution

print_class_distribution(y_train, title="BEFORE SMOTE")
X_smote, y_smote = apply_smote(X_train_full, y_train)
print_class_distribution(y_smote, title="AFTER SMOTE")

# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

pd.Series(y_train).map(dict(enumerate(target_names))).value_counts().plot(
    kind="barh", ax=axes[0], color="#3b82f6"
)
axes[0].set_title("Before SMOTE")
axes[0].set_xlabel("Count")

pd.Series(y_smote).map(dict(enumerate(target_names))).value_counts().plot(
    kind="barh", ax=axes[1], color="#22c55e"
)
axes[1].set_title("After SMOTE")
axes[1].set_xlabel("Count")

plt.tight_layout()
plt.savefig("../../figures/smote_comparison.png", dpi=150, bbox_inches="tight")
plt.show()